In [21]:
import pandas as pd
import numpy as np

In [ ]:
ball = pd.read_csv("Datasets/Cleaned_Datasets/ball_cleaned_data.csv")
matches = pd.read_csv("Datasets/Cleaned_Datasets/matches_cleaned_data.csv")

In [23]:
#DATE HANDLING
matches["match_date"] = pd.to_datetime(matches["match_date"], errors="coerce")

# Merge match_date and venue into ball data
ball = ball.merge(
    matches[["match_id", "match_date", "venue"]],
    on="match_id",
    how="left"
)

In [24]:
ball = ball.sort_values(
    by=["batter", "match_date", "match_id", "over_number", "ball_number"]
)

In [25]:
# PLAYER PER MATCH AGGREGATION
batting_match = (
    ball.groupby(
        ["match_id", "match_date", "batter", "team_batting", "team_bowling", "venue"]
    )
    .agg(
        runs_scored=("batter_runs", "sum"),
        balls_faced=("ball_number", "count"),
        fours=("batter_runs", lambda x: (x == 4).sum()),
        sixes=("batter_runs", lambda x: (x == 6).sum()),
        dismissed=("is_wicket", "max")
    )
    .reset_index()
)

In [26]:
# SORT FOR TEMPORAL FEATURES
batting_match = batting_match.sort_values(
    by=["batter", "match_date", "match_id"]
).reset_index(drop=True)

In [27]:
# PERFORMANCE METRICS
batting_match["strike_rate"] = np.where(
    batting_match["balls_faced"] > 0,
    (batting_match["runs_scored"] / batting_match["balls_faced"]) * 100,
    0
)

In [28]:
#  CAREER FEATURES
batting_match["career_matches"] = batting_match.groupby("batter").cumcount()
batting_match["career_matches"] = batting_match.groupby("batter")["career_matches"].shift(1).fillna(0).astype(int)

batting_match["career_runs"] = batting_match.groupby("batter")["runs_scored"].cumsum().shift(1).fillna(0)
batting_match["career_balls"] = batting_match.groupby("batter")["balls_faced"].cumsum().shift(1).fillna(0)

batting_match["career_avg_runs"] = np.where(
    batting_match["career_matches"] > 0,
    batting_match["career_runs"] / batting_match["career_matches"],
    0
)

batting_match["career_strike_rate"] = np.where(
    batting_match["career_balls"] > 0,
    (batting_match["career_runs"] / batting_match["career_balls"]) * 100,
    0
)

In [29]:
# RECENT FORM FEATURES (last 5 and 10 matches)

# Shifted series to avoid data leakage
batting_match["shifted_runs"] = batting_match.groupby("batter")["runs_scored"].shift(1)
batting_match["shifted_balls"] = batting_match.groupby("batter")["balls_faced"].shift(1)
batting_match["shifted_dismissed"] = batting_match.groupby("batter")["dismissed"].shift(1)

In [30]:
# Last 5 matches form
batting_match["form_runs_last_5"] = (
    batting_match.groupby("batter")["shifted_runs"]
    .rolling(window=5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

In [31]:
# Last 10 matches form
batting_match["form_runs_last_10"] = (
    batting_match.groupby("batter")["shifted_runs"]
    .rolling(window=10, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

In [32]:
# Recent strike rate: last 5 matches
def calculate_recent_sr(runs_series, balls_series, window):
    total_runs = runs_series.rolling(window=window, min_periods=1).sum()
    total_balls = balls_series.rolling(window=window, min_periods=1).sum()
    return np.where(total_balls > 0, (total_runs / total_balls) * 100, 0)

batting_match["recent_sr_last_5"] = calculate_recent_sr(
    batting_match["shifted_runs"], batting_match["shifted_balls"], 5
)

In [33]:
# Recent dismissal rate
batting_match["recent_dismissal_rate_5"] = (
    batting_match.groupby("batter")["shifted_dismissed"]
    .rolling(window=5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

In [34]:
# OPPONENT SPECIFIC FEATURE
def safe_expanding_mean(group):
    return group.shift(1).expanding().mean()

batting_match["avg_runs_vs_opponent"] = (
    batting_match.groupby(["batter", "team_bowling"])["runs_scored"]
    .transform(safe_expanding_mean)
    .fillna(0)
)

In [35]:
# VENUE-SPECIFIC FEATURE
batting_match["avg_runs_at_venue"] = (
    batting_match.groupby(["batter", "venue"])["runs_scored"]
    .transform(safe_expanding_mean)
    .fillna(0)
)

In [36]:
# BOUNDARY RATE
total_boundaries = (
    batting_match.groupby("batter")["fours"].cumsum().shift(1) +
    batting_match.groupby("batter")["sixes"].cumsum().shift(1)
).fillna(0)

batting_match["career_boundary_rate"] = np.where(
    batting_match["career_balls"] > 0,
    (total_boundaries / batting_match["career_balls"]) * 100,
    0
)

In [37]:
# TARGET VARIABLE
batting_match["target_runs"] = batting_match["runs_scored"]

In [38]:
# FINAL FEATURE SET
drop_cols = ["runs_scored", "career_runs", "career_balls", 
             "shifted_runs", "shifted_balls", "shifted_dismissed"]
features_df = batting_match.drop(columns=drop_cols)

features_df = features_df.fillna(0)

In [ ]:
# Saving file
features_df.to_csv("Datasets/Cleaned_Datasets/final_batting_features_corrected.csv", index=False)

In [41]:
features_df.head(10)

,match_id,match_date,batter,team_batting,team_bowling,venue,balls_faced,fours,sixes,dismissed,...,career_avg_runs,career_strike_rate,form_runs_last_5,form_runs_last_10,recent_sr_last_5,recent_dismissal_rate_5,avg_runs_vs_opponent,avg_runs_at_venue,career_boundary_rate,target_runs
0,548346,2012-04-29,A Ashish Reddy,Sunrisers Hyderabad,Mumbai Indians,Wankhede Stadium,10,0,1,True,...,0.000000,0.000000,0.00,0.000000,0.000000,0.00,0.0,0.000000,0.000000,10
1,548352,2012-05-04,A Ashish Reddy,Sunrisers Hyderabad,Chennai Super Kings,"MA Chidambaram Stadium, Chepauk",3,0,0,True,...,0.000000,100.000000,10.00,10.000000,100.000000,1.00,0.0,0.000000,10.000000,3
2,548359,2012-05-08,A Ashish Reddy,Sunrisers Hyderabad,Punjab Kings,"Rajiv Gandhi International Stadium, Uppal",8,1,0,True,...,13.000000,100.000000,6.50,6.500000,100.000000,1.00,0.0,0.000000,7.692308,8
3,548373,2012-05-18,A Ashish Reddy,Sunrisers Hyderabad,Rajasthan Royals,"Rajiv Gandhi International Stadium, Uppal",4,2,0,False,...,10.500000,100.000000,7.00,7.000000,100.000000,1.00,0.0,8.000000,9.523810,10
4,548376,2012-05-20,A Ashish Reddy,Sunrisers Hyderabad,Royal Challengers Bangalore,"Rajiv Gandhi International Stadium, Uppal",5,0,0,True,...,10.333333,124.000000,7.75,7.750000,124.000000,0.75,0.0,9.000000,16.000000,4
5,598000,2013-04-05,A Ashish Reddy,Sunrisers Hyderabad,Pune Warriors,"Rajiv Gandhi International Stadium, Uppal",4,1,0,False,...,8.750000,116.666667,7.00,7.000000,116.666667,0.80,0.0,7.333333,13.333333,7
6,598004,2013-04-07,A Ashish Reddy,Sunrisers Hyderabad,Royal Challengers Bangalore,"Rajiv Gandhi International Stadium, Uppal",12,0,1,True,...,8.400000,123.529412,6.40,7.000000,133.333333,0.60,4.0,7.250000,14.705882,14
7,598048,2013-04-09,A Ashish Reddy,Sunrisers Hyderabad,Royal Challengers Bangalore,M Chinnaswamy Stadium,4,0,0,True,...,9.333333,121.739130,8.60,8.000000,130.303030,0.60,9.0,0.000000,13.043478,3
8,598010,2013-04-12,A Ashish Reddy,Sunrisers Hyderabad,Delhi Capitals,Feroz Shah Kotla,9,2,0,True,...,8.428571,118.000000,7.60,7.375000,131.034483,0.60,0.0,0.000000,12.000000,16
9,598013,2013-04-14,A Ashish Reddy,Sunrisers Hyderabad,Kolkata Knight Riders,Eden Gardens,5,0,0,True,...,9.375000,127.118644,8.80,8.333333,129.411765,0.80,0.0,0.000000,13.559322,4
